<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson03/hints_2_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hints 2: Fine-tune a model on your dataset

Hints for Assignment 02, in three parts:

1. Build your own image dataset
2. **Fine-tune a model on it** ← *you are here*
3. Turn it into a web app

This is the same as [notebook 5 from lesson 02](https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/05_finetuning.ipynb), only with your own data.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
from fastai.vision.all import *

try:
    import timm
except ImportError:
    !pip install -q timm
    import timm

---
## 1. Get your dataset into Colab

Upload the ZIP from part 1 (or drag it into the file browser, the folder icon on the left) and unzip it:

In [ ]:
from google.colab import files

uploaded = files.upload()          # choose your ZIP file
zip_name = list(uploaded.keys())[0]
!unzip -q -o "{zip_name}"

*Tip:* if your ZIP is on Google Drive, you can skip the upload: run `from google.colab import drive; drive.mount("/content/drive")` and unzip it from `/content/drive/MyDrive/...`.

In [ ]:
path = Path("bears")     # TODO: the name of your dataset folder

for split in ["train", "test"]:
    for folder in sorted((path / split).iterdir()):
        if folder.is_dir():
            print(f"{split:5s} {folder.name:20s} {len(get_image_files(folder))} images")

---
## 2. The data

We use only the `train` folder for now. fastai splits it once more: 80% to train the model and 20% as the **validation set**, which we use to choose the learning rate and the number of epochs. The `test` folder stays untouched until the very end.

The same settings as in notebook 5: resize to 224×224 pixels, and data augmentation. The category of each image is the name of its folder.

In [ ]:
dls = ImageDataLoaders.from_folder(path / "train", valid_pct=0.2, seed=42,
                                   item_tfms=Resize(224),        # same size for every image
                                   batch_tfms=aug_transforms())  # data augmentation
print("categories:       ", dls.vocab)
print("training images:  ", len(dls.train_ds))
print("validation images:", len(dls.valid_ds))
dls.show_batch(max_n=8)

---
## 3. Fine-tuning

The same pretrained model as in notebook 5. (For a smaller and faster model, try `"resnet18"`.)

In [ ]:
model_name = "convnext_tiny.fb_in22k_ft_in1k"
learn = vision_learner(dls, model_name, metrics=accuracy)

suggestion = learn.lr_find()
suggestion

In [ ]:
learn.fine_tune(3, suggestion.valley)    # TODO: experiment with the number of epochs

---
## 4. Where does the model make mistakes?

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

The images with the highest loss. Wrong labels and junk images often show up here:

In [ ]:
interp.plot_top_losses(9, nrows=3)

### Optional: clean your data

The cleaner shows the images with the highest loss, for each category and for the training and validation set (choose them in the two menus). Mark the bad ones as *&lt;Delete&gt;*, or choose the correct category:

In [ ]:
from fastai.vision.widgets import ImageClassifierCleaner

cleaner = ImageClassifierCleaner(learn)
cleaner

This cell applies your choices (only for the menus selected above, so run it after each category). When you are done, run the notebook again from section 2.

In [ ]:
for index in cleaner.delete():
    cleaner.fns[index].unlink()
for index, category in cleaner.change():
    shutil.move(str(cleaner.fns[index]), path / "train" / category)

---
## 5. The test set

Only now, at the very end, we check the model on the `test` images. It has never seen them, not even when we chose the learning rate or the number of epochs. So this is an honest estimate of how well it works on new images:

In [ ]:
test_files = get_image_files(path / "test")
test_dl = dls.test_dl(test_files, with_labels=True)

loss, test_accuracy = learn.validate(dl=test_dl)
print(f"test accuracy: {test_accuracy:.1%} ({len(test_files)} images)")

In [ ]:
ClassificationInterpretation.from_learner(learn, dl=test_dl).plot_confusion_matrix()

---
## 6. Export and download the model

`export` saves the model into one file. You need it in [part 3](https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson03/hints_3_gradio.ipynb), so download it now (Colab deletes all files when the runtime disconnects):

In [ ]:
learn.export("model.pkl")
model_file = learn.path / "model.pkl"
print(model_file, f"{model_file.stat().st_size / 1e6:.0f} MB")

In [ ]:
files.download(str(model_file))